In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

DATA_FILE = 'training_data.csv'
MODEL_SAVE_PATH = 'drowsiness_model.h5'
SCALER_SAVE_PATH = 'scaler.pkl'

SEQ_LEN = 30
FEATURES = ['EAR', 'MAR', 'Pitch', 'BlinkDuration']
LABEL_COL = 'IsDrowsy'

if not os.path.exists(DATA_FILE):
    print(f"[ERROR] {DATA_FILE} not found. Please run combine.py first.")
    exit()

print(f"[INFO] Loading data from {DATA_FILE}...")
df = pd.read_csv(DATA_FILE)

# Split data into "Alert" (0) and "Drowsy" (1)
alert_df = df[df[LABEL_COL] == 0]
drowsy_df = df[df[LABEL_COL] == 1]

print(f"[INFO] Alert samples: {len(alert_df)}")
print(f"[INFO] Drowsy samples: {len(drowsy_df)}")

print("[INFO] Fitting scaler...")
scaler = StandardScaler()
X_all = df[FEATURES].values
scaler.fit(X_all)

joblib.dump(scaler, SCALER_SAVE_PATH)
print(f"[INFO] Scaler saved to {SCALER_SAVE_PATH}")

def create_sequences(dataset, seq_len):
    sequences = []
    labels = []

    data_values = scaler.transform(dataset[FEATURES].values)

    for i in range(len(data_values) - seq_len):
        seq = data_values[i : i + seq_len]
        sequences.append(seq)

        label_val = dataset.iloc[i + seq_len][LABEL_COL]
        labels.append(label_val)

    return np.array(sequences), np.array(labels)

print("[INFO] Creating sequences...")
X_alert, y_alert = create_sequences(alert_df, SEQ_LEN)
X_drowsy, y_drowsy = create_sequences(drowsy_df, SEQ_LEN)

X = np.concatenate([X_alert, X_drowsy])
y = np.concatenate([y_alert, y_drowsy])

y = to_categorical(y)

print(f"[INFO] Data Shape: {X.shape}")
print(f"[INFO] Labels Shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

print("[INFO] Building LSTM Model...")
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(FEATURES))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    # Layer 3: Output (2 neurons for 2 classes: Alert, Drowsy)
    # Activation 'softmax' gives us probabilities like [0.1, 0.9]
    Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print("[INFO] Starting training...")
epochs = 20  # You can increase this if accuracy is low
batch_size = 32

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    verbose=1
)

print(f"[INFO] Saving model to {MODEL_SAVE_PATH}...")
model.save(MODEL_SAVE_PATH)
print("[INFO] Done! Model and Scaler are ready.")

[INFO] Loading data from training_data.csv...
[INFO] Alert samples: 2065
[INFO] Drowsy samples: 1518
[INFO] Fitting scaler...
[INFO] Scaler saved to scaler.pkl
[INFO] Creating sequences...
[INFO] Data Shape: (3523, 30, 4)
[INFO] Labels Shape: (3523, 2)
[INFO] Building LSTM Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,146 (117.76 KB)

 Trainable params: 30,146 (117.76 KB)

 Non-trainable params: 0 (0.00 B)

[INFO] Starting training...
Epoch 1/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.8807 - loss: 0.3424 - val_accuracy: 0.9121 - val_loss: 0.1817
Epoch 2/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9097 - loss: 0.1657 - val_accuracy: 0.9319 - val_loss: 0.1289
Epoch 3/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9332 - loss: 0.1282 - val_accuracy: 0.9362 - val_loss: 0.1159
Epoch 4/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9351 - loss: 0.1214 - val_accuracy: 0.9433 - val_loss: 0.1277
Epoch 5/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9393 - loss: 0.1207 - val_accuracy: 0.9489 - val_loss: 0.0950
Epoch 6/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9386 - loss: 0.1175 - val_accuracy: 0.9404 - val_loss: 0.1061
Epoch 7/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9474 - loss: 0.1029 - val_accuracy: 0.9532 - val_loss: 0.0864
Epoch 8/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9435 - loss: 0.1467 - val

[INFO] Saving model to drowsiness_model.h5...
[INFO] Done! Model and Scaler are ready.
